In [1]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import seaborn as sns

from datetime import datetime
from dateutil.relativedelta import relativedelta


import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [2]:
df = pd.read_excel("../dataset/02.06.2025.xlsx")
df.head()

,App,Сумма выдачи,Дата выдачи,Дата погашения,Срок кредита (месяц),Статус кредита,Кредитный портфель#,Кредитный портфель вал.#,Портфель/Итог(%)#,Остаток стандарт#,...,Страна,Область \город,Регион \город,Село,Адрес месторождения,Семейное положение,Количество членов семьи,Образование,Место работы,Должность
0,C-00-30283,8000000.0,18.07.2018,17.07.2020,24.0,Списанный,3346360.8,0,0.0025,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,ФУРКАТ ТУМАНИ,NaN,NaN,NaN,1.0,Средне-специальное,Фуркат тумани Агросаноат ва транспорт кхк,хисобчи
1,C-00-30877,3000000.0,29.08.2018,05.08.2019,11.0,Списанный,824897.69,0,0.0006,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалиёт,ёрдамчи
2,C-00-31099,3000000.0,18.09.2018,18.09.2019,12.0,Списанный,622134.54,0,0.0004,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,КУКОН ШАХРИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалиёт,ёрдамчи
3,C-00-31145,3000000.0,20.09.2018,20.09.2019,12.0,Списанный,549419.14,0,0.0004,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,КУКОН ШАХРИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалиёт,шофёр
4,C-00-30609,3000000.0,13.08.2018,13.08.2019,12.0,Списанный,1722613.68,0,0.0012,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалиёт,нафакахур


In [3]:
df.columns = [col.lower() for col in df.columns]

In [4]:
df["кредитный продукт"].value_counts()

кредитный продукт
ISHONCH           7068
KAPITAL           3842
BARAKA             627
KAFOLAT            447
HAMKOR KAPITAL     142
Сармоя              85
Хамкор              54
Хазина              20
Хамкор Сармоя       14
Яхши ният            4
Мадор                4
Имкон                2
Name: count, dtype: int64

In [5]:
df_no_assets = df[df["кредитный продукт"] == 'ISHONCH']
df_no_assets.head()

,app,сумма выдачи,дата выдачи,дата погашения,срок кредита (месяц),статус кредита,кредитный портфель#,кредитный портфель вал.#,портфель/итог(%)#,остаток стандарт#,...,страна,область \город,регион \город,село,адрес месторождения,семейное положение,количество членов семьи,образование,место работы,должность
143,C-00-52877,5000000.0,11.08.2023,09.08.2024,12.0,Списанный,3746309.4,0,0.0028,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,xususiy amalyot,уста
212,C-00-53282,5000000.0,05.09.2023,05.09.2024,12.0,Списанный,4580890.3,0,0.0034,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,ФУРКАТ ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,xususiy amaliyot,уста
214,C-00-53012,4000000.0,18.08.2023,16.08.2024,12.0,Списанный,478947.03,0,0.0003,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,xususiy amaliyot,богбон
261,C-00-53277,5000000.0,04.09.2023,04.09.2024,12.0,Списанный,319060.65,0,0.0002,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,КУВА ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалийёт,Шифокор
282,C-00-53533,4000000.0,13.09.2023,13.09.2024,12.0,Списанный,2673022.36,0,0.0020,0,...,Узбекистан,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,Xususiy amaliyot,уста


In [6]:
len(df_no_assets)

7068

In [7]:
df_no_assets["кол-во просроч-х дней#"].isna().sum()

0

In [8]:
df_no_assets["кол-во просроч-х дней#"] = (
    df_no_assets["кол-во просроч-х дней#"]
    .astype(str)                     
    .str.replace(r"[^\d.]", "", regex=True)  
    .astype(float)                   
    .astype(int)                  
)

In [9]:
df_no_assets["кол-во просроч-х дней#"].dtype

dtype('int64')

In [10]:
df_no_assets.dropna(subset=["статус кредита"], inplace=True)

In [11]:
df_no_assets['статус кредита'].value_counts()

статус кредита
Активный     6993
Списанный      75
Name: count, dtype: int64

In [12]:
df_copy = df_no_assets.copy()
df_copy.columns

Index(['app', 'сумма выдачи', 'дата выдачи', 'дата погашения',
       'срок кредита (месяц)', 'статус кредита', 'кредитный портфель#',
       'кредитный портфель вал.#', 'портфель/итог(%)#', 'остаток стандарт#',
       'остаток стандарт вал.#', 'остаток просрочки#',
       'остаток просрочки вал.#', 'остаток пролонг#', 'остаток пролонг вал#',
       'остаток пролонг просроч.#', 'остаток пролонг просроч. вал.#',
       'остаток судопро-во#', 'остаток судопро-во вал.#',
       'лицевой счет процент', 'начисленные проценты#',
       'лицевой счет процент. внесис-й.', 'начисленные проценты внесис-й#',
       'лиц.счет текущего счета', 'остаток на тек.счете#', 'цикл',
       'кредитный продукт', 'цель кредита', 'процентная ставка',
       'дата просрочки', 'кол-во просроч-х дней#',
       'кол-во просроч-х дней по ос#', 'кол-во просроч-х дней по %#',
       'просроченная сумма по ос #', 'просроченная сумма по % #',
       'общ. кол. дни просрочки #', 'макс-я. кол. дни просрочки',
       'об

In [14]:
df_copy['дата выдачи'] = pd.to_datetime(df_copy['дата выдачи'], format='%d.%m.%Y')
df_copy['дата выдачи']

143     2023-08-11
212     2023-09-05
214     2023-08-18
261     2023-09-04
282     2023-09-13
           ...    
12300   2025-06-02
12302   2025-06-02
12303   2025-06-02
12305   2025-06-02
12306   2025-06-02
Name: дата выдачи, Length: 7068, dtype: datetime64[ns]

In [15]:
current_date = pd.to_datetime('02.06.2025', format='%d.%m.%Y')
current_date

Timestamp('2025-06-02 00:00:00')

In [16]:
df_copy["дата выдачи"] = pd.to_datetime(df_copy["дата выдачи"], dayfirst=True)
df_copy["дата выдачи"]

143     2023-08-11
212     2023-09-05
214     2023-08-18
261     2023-09-04
282     2023-09-13
           ...    
12300   2025-06-02
12302   2025-06-02
12303   2025-06-02
12305   2025-06-02
12306   2025-06-02
Name: дата выдачи, Length: 7068, dtype: datetime64[ns]

In [17]:
df_copy["months_diff"] = (current_date.year - df_copy["дата выдачи"].dt.year) * 12 + (current_date.month - df_copy["дата выдачи"].dt.month)
df_copy["months_diff"]

143      22
212      21
214      22
261      21
282      21
         ..
12300     0
12302     0
12303     0
12305     0
12306     0
Name: months_diff, Length: 7068, dtype: int32

In [18]:
df_matured = df_copy[df_copy["months_diff"] >= 6]

In [19]:
df_matured.head()

,app,сумма выдачи,дата выдачи,дата погашения,срок кредита (месяц),статус кредита,кредитный портфель#,кредитный портфель вал.#,портфель/итог(%)#,остаток стандарт#,...,область \город,регион \город,село,адрес месторождения,семейное положение,количество членов семьи,образование,место работы,должность,months_diff
143,C-00-52877,5000000.0,2023-08-11,09.08.2024,12.0,Списанный,3746309.4,0,0.0028,0,...,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,xususiy amalyot,уста,22
212,C-00-53282,5000000.0,2023-09-05,05.09.2024,12.0,Списанный,4580890.3,0,0.0034,0,...,ФАРГОНА ВИЛОЯТИ,ФУРКАТ ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,xususiy amaliyot,уста,21
214,C-00-53012,4000000.0,2023-08-18,16.08.2024,12.0,Списанный,478947.03,0,0.0003,0,...,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,xususiy amaliyot,богбон,22
261,C-00-53277,5000000.0,2023-09-04,04.09.2024,12.0,Списанный,319060.65,0,0.0002,0,...,ФАРГОНА ВИЛОЯТИ,КУВА ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалийёт,Шифокор,21
282,C-00-53533,4000000.0,2023-09-13,13.09.2024,12.0,Списанный,2673022.36,0,0.0020,0,...,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,Xususiy amaliyot,уста,21


In [20]:
overdue_col = "кол-во просроч-х дней#"

In [21]:
df_matured["is_bad_client"] = (
    (df_matured[overdue_col] >= 60) | 
    (df_matured["статус кредита"].str.lower() == "списанный")
)

In [22]:
df_matured["is_bad_client"].value_counts()

is_bad_client
False    3289
True      384
Name: count, dtype: int64

In [23]:
npl_ratio = df_matured["is_bad_client"].mean() * 100
print(f"NPL: {npl_ratio:.2f}%")

NPL: 10.45%


In [24]:
df_matured.head()

,app,сумма выдачи,дата выдачи,дата погашения,срок кредита (месяц),статус кредита,кредитный портфель#,кредитный портфель вал.#,портфель/итог(%)#,остаток стандарт#,...,регион \город,село,адрес месторождения,семейное положение,количество членов семьи,образование,место работы,должность,months_diff,is_bad_client
143,C-00-52877,5000000.0,2023-08-11,09.08.2024,12.0,Списанный,3746309.4,0,0.0028,0,...,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,xususiy amalyot,уста,22,True
212,C-00-53282,5000000.0,2023-09-05,05.09.2024,12.0,Списанный,4580890.3,0,0.0034,0,...,ФУРКАТ ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,xususiy amaliyot,уста,21,True
214,C-00-53012,4000000.0,2023-08-18,16.08.2024,12.0,Списанный,478947.03,0,0.0003,0,...,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,xususiy amaliyot,богбон,22,True
261,C-00-53277,5000000.0,2023-09-04,04.09.2024,12.0,Списанный,319060.65,0,0.0002,0,...,КУВА ТУМАНИ,NaN,NaN,NaN,1.0,Среднее,хусусий амалийёт,Шифокор,21,True
282,C-00-53533,4000000.0,2023-09-13,13.09.2024,12.0,Списанный,2673022.36,0,0.0020,0,...,УЗБЕКИСТОН ТУМАНИ,NaN,NaN,NaN,NaN,Среднее,Xususiy amaliyot,уста,21,True


In [25]:
df_matured.columns

Index(['app', 'сумма выдачи', 'дата выдачи', 'дата погашения',
       'срок кредита (месяц)', 'статус кредита', 'кредитный портфель#',
       'кредитный портфель вал.#', 'портфель/итог(%)#', 'остаток стандарт#',
       'остаток стандарт вал.#', 'остаток просрочки#',
       'остаток просрочки вал.#', 'остаток пролонг#', 'остаток пролонг вал#',
       'остаток пролонг просроч.#', 'остаток пролонг просроч. вал.#',
       'остаток судопро-во#', 'остаток судопро-во вал.#',
       'лицевой счет процент', 'начисленные проценты#',
       'лицевой счет процент. внесис-й.', 'начисленные проценты внесис-й#',
       'лиц.счет текущего счета', 'остаток на тек.счете#', 'цикл',
       'кредитный продукт', 'цель кредита', 'процентная ставка',
       'дата просрочки', 'кол-во просроч-х дней#',
       'кол-во просроч-х дней по ос#', 'кол-во просроч-х дней по %#',
       'просроченная сумма по ос #', 'просроченная сумма по % #',
       'общ. кол. дни просрочки #', 'макс-я. кол. дни просрочки',
       'об

In [26]:
selected_features = [
    'дата выдачи', 'семейное положение',
    'сумма выдачи', 'кредитный портфель#' ,'срок кредита (месяц)', 'цикл', 'цель кредита', 'процентная ставка', 
    'возраст', 'пол', 'область \город', 'регион \город','количество членов семьи',
    'образование', 'должность', 'is_bad_client'
]

In [27]:
selected = df_matured[selected_features]
selected.head()

,дата выдачи,семейное положение,сумма выдачи,кредитный портфель#,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,область \город,регион \город,количество членов семьи,образование,должность,is_bad_client
143,2023-08-11,NaN,5000000.0,3746309.4,12.0,1.0,Миграция,72.0,47.0,М,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,Среднее,уста,True
212,2023-09-05,NaN,5000000.0,4580890.3,12.0,1.0,Миграция,72.0,28.0,М,ФАРГОНА ВИЛОЯТИ,ФУРКАТ ТУМАНИ,NaN,Среднее,уста,True
214,2023-08-18,NaN,4000000.0,478947.03,12.0,1.0,Миграция,72.0,30.0,Ж,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,1.0,Среднее,богбон,True
261,2023-09-04,NaN,5000000.0,319060.65,12.0,1.0,Миграция,72.0,40.0,М,ФАРГОНА ВИЛОЯТИ,КУВА ТУМАНИ,1.0,Среднее,Шифокор,True
282,2023-09-13,NaN,4000000.0,2673022.36,12.0,1.0,Миграция,72.0,22.0,М,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,Среднее,уста,True


In [28]:
selected.rename(columns={'кредитный портфель#': 'остаток суммы'}, inplace=True)
selected.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,область \город,регион \город,количество членов семьи,образование,должность,is_bad_client
143,2023-08-11,NaN,5000000.0,3746309.4,12.0,1.0,Миграция,72.0,47.0,М,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,Среднее,уста,True
212,2023-09-05,NaN,5000000.0,4580890.3,12.0,1.0,Миграция,72.0,28.0,М,ФАРГОНА ВИЛОЯТИ,ФУРКАТ ТУМАНИ,NaN,Среднее,уста,True
214,2023-08-18,NaN,4000000.0,478947.03,12.0,1.0,Миграция,72.0,30.0,Ж,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,1.0,Среднее,богбон,True
261,2023-09-04,NaN,5000000.0,319060.65,12.0,1.0,Миграция,72.0,40.0,М,ФАРГОНА ВИЛОЯТИ,КУВА ТУМАНИ,1.0,Среднее,Шифокор,True
282,2023-09-13,NaN,4000000.0,2673022.36,12.0,1.0,Миграция,72.0,22.0,М,ФАРГОНА ВИЛОЯТИ,УЗБЕКИСТОН ТУМАНИ,NaN,Среднее,уста,True


In [29]:
selected.isna().sum()

дата выдачи                   0
семейное положение         1132
сумма выдачи                  0
остаток суммы                 0
срок кредита (месяц)          0
цикл                          0
цель кредита                  0
процентная ставка             0
возраст                       0
пол                           0
область \город                0
регион \город                 0
количество членов семьи      56
образование                 321
должность                   488
is_bad_client                 0
dtype: int64

In [30]:
selected['дата выдачи'] = pd.to_datetime(selected['дата выдачи'], dayfirst=True, errors='coerce')

In [31]:
selected['year'] = selected['дата выдачи'].dt.year

In [32]:
obj_df=selected.select_dtypes(include=['object']).copy()

In [33]:
def find_unique(col):
    print(col,":",obj_df[col].unique())

In [34]:
for col in obj_df.columns:
  find_unique(col)

семейное положение : [nan 'Женат \\ замужем' 'Холост \\ незамужем' 'Рахимов Юсуфжон'
 'В разводе' 'Вдовец \\ вдова' 'Mirzayeva Gulshanoy']
остаток суммы : [3746309.4 4580890.3 478947.03 ... 7799197.59 4082987.9 7810148.67]
цель кредита : ['Миграция' 'Деҳқончилик, томорқа хўжалигини ривожлантириш'
 'Таъмирлаш ишлари учун  (оилавий тадбирлар, таълим, даволаниш)'
 'Оилавий ва шахсий эҳтиёжлар (оилавий тадбирлар, таълим, даволаниш)'
 'Чорвачиликни ривожлантириш'
 'Ҳунармандчилик, касаначиликни ривожлантириш'
 'Энергия тежовчи ускуналар сотиб олиш, ўрнатиш'
 'Ички туризмни ривожлантириш' 'Савдо' 'Хизмат курсатиш' 'Ишлаб чикариш']
пол : ['М' 'Ж']
область \город : ['ФАРГОНА ВИЛОЯТИ' 'НАМАНГАН ВИЛОЯТИ' 'АНДИЖОН ВИЛОЯТИ' 'ТОШКЕНТ ШАХРИ'
 'ТОШКЕНТ ВИЛОЯТИ']
регион \город : ['УЗБЕКИСТОН ТУМАНИ' 'ФУРКАТ ТУМАНИ' 'КУВА ТУМАНИ' 'ЧУСТ ТУМАНИ'
 'КУКОН ШАХРИ' 'ДАНГАРА ТУМАНИ' 'УЧКУПРИК ТУМАНИ' 'ТУРАКУРГОН ТУМАНИ'
 'ОЛТИАРИК ТУМАНИ' 'ПОП ТУМАНИ' 'УЧКУРГОН ТУМАНИ' 'ЯНГИКУРГОН ТУМАНИ'
 'БУВАЙДА ТУМАНИ' 'МИ

In [35]:
selected.columns

Index(['дата выдачи', 'семейное положение', 'сумма выдачи', 'остаток суммы',
       'срок кредита (месяц)', 'цикл', 'цель кредита', 'процентная ставка',
       'возраст', 'пол', 'область \город', 'регион \город',
       'количество членов семьи', 'образование', 'должность', 'is_bad_client',
       'year'],
      dtype='object')

In [36]:
cols_to_clean = [
    'цель кредита',
    'пол', 'образование', 'должность', 'область \город',
       'регион \город' 
]

In [37]:
for col in cols_to_clean:
    selected[col] = selected[col].astype(str).str.strip().str.lower()

In [38]:
bad_values = {
    'образование': [
        'o‘qituvchi 25-maktab',
        'xususiy amaliyot'
    ]
}

In [39]:
for col, values in bad_values.items():
    selected = selected[~selected[col].astype(str).str.strip().str.lower().isin([v.lower() for v in values])]

selected = selected.reset_index(drop=True)

In [40]:
obj_df=selected.select_dtypes(include=['object']).copy()

In [41]:
for col in obj_df.columns:
    find_unique(col)

семейное положение : [nan 'Женат \\ замужем' 'Холост \\ незамужем' 'Рахимов Юсуфжон'
 'В разводе' 'Вдовец \\ вдова' 'Mirzayeva Gulshanoy']
остаток суммы : [3746309.4 4580890.3 478947.03 ... 7799197.59 4082987.9 7810148.67]
цель кредита : ['миграция' 'деҳқончилик, томорқа хўжалигини ривожлантириш'
 'таъмирлаш ишлари учун  (оилавий тадбирлар, таълим, даволаниш)'
 'оилавий ва шахсий эҳтиёжлар (оилавий тадбирлар, таълим, даволаниш)'
 'чорвачиликни ривожлантириш'
 'ҳунармандчилик, касаначиликни ривожлантириш'
 'энергия тежовчи ускуналар сотиб олиш, ўрнатиш'
 'ички туризмни ривожлантириш' 'савдо' 'хизмат курсатиш' 'ишлаб чикариш']
пол : ['м' 'ж']
область \город : ['фаргона вилояти' 'наманган вилояти' 'андижон вилояти' 'тошкент шахри'
 'тошкент вилояти']
регион \город : ['узбекистон тумани' 'фуркат тумани' 'кува тумани' 'чуст тумани'
 'кукон шахри' 'дангара тумани' 'учкуприк тумани' 'туракургон тумани'
 'олтиарик тумани' 'поп тумани' 'учкургон тумани' 'янгикургон тумани'
 'бувайда тумани' 'ми

In [42]:
selected['образование'].value_counts()

образование
среднее                2103
средне-специальное     1032
nan                     321
высшее                  128
неоконченное высшее      45
среднее_x000d_           41
урта махсус               1
урта-махсус               1
начальное                 1
Name: count, dtype: int64

In [43]:
selected['образование'] = selected['образование'].replace({
        'nan': 'урта махсус '
    })

In [44]:
import re
def clean_text_columns_strict(df, columns):
    
    for col in columns:
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value).astype(str).str.lower()
        df[col] = df[col].apply(lambda x: re.sub(r'[^a-zа-яё0-9\s]', '', x))
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()

    return df

In [45]:
selected = clean_text_columns_strict(selected, ['образование', 'должность'])

In [46]:
selected["образование"].value_counts()

образование
среднее                2103
среднеспециальное      1032
урта махсус             322
высшее                  128
неоконченное высшее      45
среднееx000d             41
уртамахсус                1
начальное                 1
Name: count, dtype: int64

In [47]:
selected['образование'] = selected['образование'].replace({
    'уртамахсус': 'среднее',
    'урта': 'среднее',
    'урта махсус': 'среднее',
    'среднее_x000d_\n': 'среднее',
    'xususiy amaliyot': 'другое',
    'неоконченное высшее': 'высшее',
    'среднеспециальное': 'среднее',
    'среднееx000d': 'среднее',
    'средне': 'среднее',
    'среднее_x000d_': 'среднее',
    'o‘qituvchi 25-maktab': 'высшее',
    'nan': 'начальное',
    'неоконченное высшее': 'высшее'

})

In [48]:
selected['образование'].value_counts()

образование
среднее      3499
высшее        173
начальное       1
Name: count, dtype: int64

In [49]:
selected.dropna(subset=["образование"], inplace=True)

In [50]:
selected["должность"].value_counts()

должность
nan                 488
тикувчи             281
савдогар            175
хайдовчи            170
уста                118
                   ... 
кондитер              1
нафакада сотувчи      1
сотувчитикувчи        1
mebel uyda            1
dehqon                1
Name: count, Length: 571, dtype: int64

In [51]:
selected['должность'] = selected['должность'].replace({
    'nan': 'уста',
})

In [52]:
selected["должность"].value_counts()

должность
уста                606
тикувчи             281
савдогар            175
хайдовчи            170
sotuvchi            105
                   ... 
кондитер              1
нафакада сотувчи      1
сотувчитикувчи        1
mebel uyda            1
dehqon                1
Name: count, Length: 570, dtype: int64

In [53]:
selected.dropna(subset=["должность"], inplace=True)

In [54]:
df_cleaned = selected.copy(deep=True)
df_cleaned.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,область \город,регион \город,количество членов семьи,образование,должность,is_bad_client,year
0,2023-08-11,NaN,5000000.0,3746309.4,12.0,1.0,миграция,72.0,47.0,м,фаргона вилояти,узбекистон тумани,NaN,среднее,уста,True,2023
1,2023-09-05,NaN,5000000.0,4580890.3,12.0,1.0,миграция,72.0,28.0,м,фаргона вилояти,фуркат тумани,NaN,среднее,уста,True,2023
2,2023-08-18,NaN,4000000.0,478947.03,12.0,1.0,миграция,72.0,30.0,ж,фаргона вилояти,узбекистон тумани,1.0,среднее,богбон,True,2023
3,2023-09-04,NaN,5000000.0,319060.65,12.0,1.0,миграция,72.0,40.0,м,фаргона вилояти,кува тумани,1.0,среднее,шифокор,True,2023
4,2023-09-13,NaN,4000000.0,2673022.36,12.0,1.0,миграция,72.0,22.0,м,фаргона вилояти,узбекистон тумани,NaN,среднее,уста,True,2023


In [55]:
df_cleaned['количество членов семьи'].isna().sum()

56

In [56]:
mode_value = df_cleaned['количество членов семьи'].mode()[0]
df_cleaned['количество членов семьи'] = df_cleaned['количество членов семьи'].fillna(mode_value)

In [57]:
df_cleaned['семейное положение'] = np.where(df_cleaned['количество членов семьи'] == 1, 'single', 'married')

In [58]:
df_cleaned.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,область \город,регион \город,количество членов семьи,образование,должность,is_bad_client,year
0,2023-08-11,married,5000000.0,3746309.4,12.0,1.0,миграция,72.0,47.0,м,фаргона вилояти,узбекистон тумани,4.0,среднее,уста,True,2023
1,2023-09-05,married,5000000.0,4580890.3,12.0,1.0,миграция,72.0,28.0,м,фаргона вилояти,фуркат тумани,4.0,среднее,уста,True,2023
2,2023-08-18,single,4000000.0,478947.03,12.0,1.0,миграция,72.0,30.0,ж,фаргона вилояти,узбекистон тумани,1.0,среднее,богбон,True,2023
3,2023-09-04,single,5000000.0,319060.65,12.0,1.0,миграция,72.0,40.0,м,фаргона вилояти,кува тумани,1.0,среднее,шифокор,True,2023
4,2023-09-13,married,4000000.0,2673022.36,12.0,1.0,миграция,72.0,22.0,м,фаргона вилояти,узбекистон тумани,4.0,среднее,уста,True,2023


In [59]:
df_cleaned['is_bad_client'].value_counts()

is_bad_client
False    3289
True      384
Name: count, dtype: int64

In [60]:
df_cleaned['is_bad_client'] = df_cleaned['is_bad_client'].astype(int)

In [61]:
df_cleaned.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,область \город,регион \город,количество членов семьи,образование,должность,is_bad_client,year
0,2023-08-11,married,5000000.0,3746309.4,12.0,1.0,миграция,72.0,47.0,м,фаргона вилояти,узбекистон тумани,4.0,среднее,уста,1,2023
1,2023-09-05,married,5000000.0,4580890.3,12.0,1.0,миграция,72.0,28.0,м,фаргона вилояти,фуркат тумани,4.0,среднее,уста,1,2023
2,2023-08-18,single,4000000.0,478947.03,12.0,1.0,миграция,72.0,30.0,ж,фаргона вилояти,узбекистон тумани,1.0,среднее,богбон,1,2023
3,2023-09-04,single,5000000.0,319060.65,12.0,1.0,миграция,72.0,40.0,м,фаргона вилояти,кува тумани,1.0,среднее,шифокор,1,2023
4,2023-09-13,married,4000000.0,2673022.36,12.0,1.0,миграция,72.0,22.0,м,фаргона вилояти,узбекистон тумани,4.0,среднее,уста,1,2023


In [62]:
df_cleaned.isna().sum()

дата выдачи                0
семейное положение         0
сумма выдачи               0
остаток суммы              0
срок кредита (месяц)       0
цикл                       0
цель кредита               0
процентная ставка          0
возраст                    0
пол                        0
область \город             0
регион \город              0
количество членов семьи    0
образование                0
должность                  0
is_bad_client              0
year                       0
dtype: int64

In [63]:
import requests

unique_years = df_cleaned['year'].unique()

#year_to_rate mapping
year_to_rate = {}

for year in unique_years:
    date_str = f"{year}-02-01"
    url = f"https://cbu.uz/uz/arkhiv-kursov-valyut/json/all/{date_str}/"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        for item in data:
            if item['Ccy'] == 'USD':
                year_to_rate[year] = int(float(item['Rate']))
                break
    else:
        year_to_rate[year] = None 


df_cleaned['usd_rate'] = df_cleaned['year'].map(year_to_rate)


In [64]:
df_cleaned['область \город'].value_counts()

область \город
фаргона вилояти     1773
наманган вилояти    1516
андижон вилояти      381
тошкент шахри          2
тошкент вилояти        1
Name: count, dtype: int64

In [66]:
shaped_bread_prices = {
    'фаргона вилояти': 2650,
    'наманган вилояти': 2700,
    'андижон вилояти': 2700,
    'тошкент шахри': 2800,
    'тошкент вилояти': 2800
}

cottonseed_oil_prices = {
    'фаргона вилояти': 19750,
    'наманган вилояти': 17100,
    'андижон вилояти': 18500,
    'тошкент шахри': 18250,
    'тошкент вилояти': 18250
}

beef_prices = {
    'фаргона вилояти': 77000,
    'наманган вилояти': 76000,
    'андижон вилояти': 82500,
    'тошкент шахри': 85000,
    'тошкент вилояти': 85000
}

In [67]:
df_cleaned['shaped_bread(som)'] = df_cleaned['область \\город'].map(shaped_bread_prices)
df_cleaned['cottonseed oil(som)'] = df_cleaned['область \\город'].map(cottonseed_oil_prices)
df_cleaned['beef(som)'] = df_cleaned['область \\город'].map(beef_prices)

In [68]:
df_cleaned.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,...,регион \город,количество членов семьи,образование,должность,is_bad_client,year,usd_rate,shaped_bread(som),cottonseed oil(som),beef(som)
0,2023-08-11,married,5000000.0,3746309.4,12.0,1.0,миграция,72.0,47.0,м,...,узбекистон тумани,4.0,среднее,уста,1,2023,11269,2650,19750,77000
1,2023-09-05,married,5000000.0,4580890.3,12.0,1.0,миграция,72.0,28.0,м,...,фуркат тумани,4.0,среднее,уста,1,2023,11269,2650,19750,77000
2,2023-08-18,single,4000000.0,478947.03,12.0,1.0,миграция,72.0,30.0,ж,...,узбекистон тумани,1.0,среднее,богбон,1,2023,11269,2650,19750,77000
3,2023-09-04,single,5000000.0,319060.65,12.0,1.0,миграция,72.0,40.0,м,...,кува тумани,1.0,среднее,шифокор,1,2023,11269,2650,19750,77000
4,2023-09-13,married,4000000.0,2673022.36,12.0,1.0,миграция,72.0,22.0,м,...,узбекистон тумани,4.0,среднее,уста,1,2023,11269,2650,19750,77000


In [69]:
df_cleaned['сумма_выдачи(usd)'] = (df_cleaned['сумма выдачи'] / df_cleaned['usd_rate']).astype(int)
df_cleaned['остаток_суммы(usd)'] = (df_cleaned['остаток суммы'] / df_cleaned['usd_rate']).astype(int)

In [70]:
df_cleaned.head()

,дата выдачи,семейное положение,сумма выдачи,остаток суммы,срок кредита (месяц),цикл,цель кредита,процентная ставка,возраст,пол,...,образование,должность,is_bad_client,year,usd_rate,shaped_bread(som),cottonseed oil(som),beef(som),сумма_выдачи(usd),остаток_суммы(usd)
0,2023-08-11,married,5000000.0,3746309.4,12.0,1.0,миграция,72.0,47.0,м,...,среднее,уста,1,2023,11269,2650,19750,77000,443,332
1,2023-09-05,married,5000000.0,4580890.3,12.0,1.0,миграция,72.0,28.0,м,...,среднее,уста,1,2023,11269,2650,19750,77000,443,406
2,2023-08-18,single,4000000.0,478947.03,12.0,1.0,миграция,72.0,30.0,ж,...,среднее,богбон,1,2023,11269,2650,19750,77000,354,42
3,2023-09-04,single,5000000.0,319060.65,12.0,1.0,миграция,72.0,40.0,м,...,среднее,шифокор,1,2023,11269,2650,19750,77000,443,28
4,2023-09-13,married,4000000.0,2673022.36,12.0,1.0,миграция,72.0,22.0,м,...,среднее,уста,1,2023,11269,2650,19750,77000,354,237


In [71]:
df_cleaned.to_csv("../RnD/df_6month_60difference.csv", index=False)
# df_cleaned.to_csv("df2307.csv", index=False)